# 14.9 - Multi-Agent Systems

Status: VERIFIED

## What Are We Solving?
Some tasks require different expertise: one agent researches, another writes, a third reviews. Coordination between agents enables decomposition and quality control.

In [1]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # loads from .env in project root
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "qwen/qwen3.8-27b"

# Quick test
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'groq ok' only"}],
    max_tokens=10
)
print(f"Groq connected: {r.choices[0].message.content.strip()}")
print(f"Model: {MODEL}")

Groq connected: groq ok
Model: qwen/qwen3.8-27b


## Specialist + Orchestrator Pattern

In [2]:
class SpecialistAgent:
    def __init__(self, name: str, system_prompt: str):
        self.name = name
        self.system_prompt = system_prompt
    
    def run(self, task: str) -> str:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": task}
            ]
        )
        return response.choices[0].message.content

class OrchestratorAgent:
    def __init__(self):
        self.specialists = {}
    
    def add_specialist(self, name: str, agent: SpecialistAgent):
        self.specialists[name] = agent
    
    def decompose(self, task: str) -> list:
        specialist_list = ", ".join(self.specialists.keys())
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": (
                    f"Break this task into subtasks. Available specialists: {specialist_list}\n"
                    "Return a JSON array: [{'specialist': 'name', 'task': 'description'}]"
                )},
                {"role": "user", "content": task}
            ]
        )
        text = response.choices[0].message.content
        # Parse JSON from response
        import re
        json_match = re.search(r'\[.*\]', text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
        return [{"specialist": list(self.specialists.keys())[0], "task": task}]
    
    def run(self, task: str) -> dict:
        plan = self.decompose(task)
        results = {}
        
        print("Orchestration plan:")
        for item in plan:
            name = item["specialist"]
            subtask = item["task"]
            print(f"  -> {name}: {subtask[:60]}")
            
            if name in self.specialists:
                result = self.specialists[name].run(subtask)
                results[name] = result[:200]
        
        return results

# Create specialists
orchestrator = OrchestratorAgent()
orchestrator.add_specialist("researcher", SpecialistAgent(
    "Researcher", "You are a research specialist. Find and summarize information concisely."
))
orchestrator.add_specialist("writer", SpecialistAgent(
    "Writer", "You are a writing specialist. Create clear, well-structured content."
))

results = orchestrator.run("Write a brief guide on Python decorators")
for name, result in results.items():
    print(f"\n--- {name} ---\n{result[:150]}...")

Orchestration plan:
  -> researcher: Research core concepts of Python decorators: syntax, functio


  -> writer: Write a brief, beginner-friendly guide on Python decorators 



--- researcher ---
# Python Decorators: Core Concepts

## 1. Syntax

A decorator is a callable object that takes a function (or class) as an argument and returns a modif...

--- writer ---
# Understanding Python Decorators: A Beginner’s Guide

Decorators are one of Python’s most powerful features, allowing you to extend the functionality...


In [3]:
# Verification
assert len(results) > 0, "Must have results from specialists"
print("VERIFICATION PASSED: Phase 14.9 complete")

VERIFICATION PASSED: Phase 14.9 complete
